In [0]:
# COMMAND ----------
%load_ext autoreload
%autoreload 2

import sys

# Disable generation of old .pyc cache files .pyc
sys.dont_write_bytecode = True

In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # 🕵️ Advanced Performance Auditor (Serverless Safe)
# MAGIC Notebook orchestrating heuristic performance and FinOps audits for **Bronze** layer using the engine `PerformanceEngine`.

In [0]:
# COMMAND ----------
import sys
import os

# Dynamically attach project root directory to sys.path
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Imports of APM engine components
from src.auditor.engine import PerformanceEngine
from src.readers.dataframe_reader import DataFrameExplainReader
from src.policies.policy_manager import PolicyManager
from src.context.environment_provider import EnvironmentProvider
from src.suggestions.remediation_engine import RemediationEngine
from src.finops.cost_translator import CostTranslator
from src.reporters.console_reporter import ConsoleReporter

# Imports of all detection rules
from src.rules.physical_rules import (
    SmallFilesRule,
    MissedBroadcastRule,
    OverPartitioningRule,
    DataSkewRule,  # <-- NEW (PERF-006)
    MissingOptimizationRule,  # <-- NEW (PERF-007)
)

# Imports of logical/code rules
from src.rules.query_rules import (
    TypeCastingRule,
    CartesianProductRule,
    ExplodeAbuseRule,
    RedundantScanRule,
    NonVectorizedUdfRule,
)

# 1. Inicjalizacja globalnego stosu technologicznego (Dependency injection)
policy_mgr = PolicyManager()
env_prov = EnvironmentProvider(spark)
remedy_eng = RemediationEngine()
cost_trans = CostTranslator(policy_mgr.get_policy("finops"))
reporter = ConsoleReporter()

# 2. Assembling complete set of active performance detectives
active_rules = [
    SmallFilesRule(),  # PERF-001
    MissedBroadcastRule(),  # PERF-002
    TypeCastingRule(),  # PERF-003
    OverPartitioningRule(),  # PERF-004
    CartesianProductRule(),  # PERF-005
    DataSkewRule(),  # PERF-006
    MissingOptimizationRule(),  # PERF-007
    ExplodeAbuseRule(),  # PERF-008
    RedundantScanRule(),  # PERF-009
    NonVectorizedUdfRule(),  # PERF-010
]

print(
    f"🔥 [APM CORE] Silnik uzbrojony w kompletny pakiet {len(active_rules)} heuristic rules. System ready for full diagnostics."
)

In [0]:
# COMMAND ----------
print("🕵️ Starting audit for process: Ingest telemetrii BESS (ETL-01)...")

TABLE_BESS = "workspace.bronze.bess_telemetry_small_files"
BESS_SOURCE_PATH = "/Volumes/workspace/default/weather_data/raw_source/bess_telemetry_raw"

# Loading data in-memory
df_bess = spark.read.table(TABLE_BESS)

# Reader receives DataFrame and physical file path
reader_bess = DataFrameExplainReader(spark, df=df_bess, source_volume_path=BESS_SOURCE_PATH)

engine_bess = PerformanceEngine(
    reader_bess, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_bess.run_audit(context_name="ETL-01-BESS-Telemetry")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Dictionary enrichment (ETL-03)...")

TABLE_ENRICHED = "workspace.bronze.enriched_telemetry_heavy_shuffle"
df_enriched = spark.read.table(TABLE_ENRICHED)

# Pass df for schema and table_name for direct SQL EXPLAIN query under Serverless
reader_enriched = DataFrameExplainReader(spark, df=df_enriched, table_name=TABLE_ENRICHED)

engine_enriched = PerformanceEngine(
    reader_enriched, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_enriched.run_audit(context_name="ETL-03-Station-Enrichment")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: PV readings ingest (ETL-02)...")

TABLE_PV = "workspace.bronze.pv_metrics_string_nightmare"
df_pv = spark.read.table(TABLE_PV)

# Safe parameter passing for structural schema analysis
reader_pv = DataFrameExplainReader(spark, df=df_pv, table_name=TABLE_PV)

engine_pv = PerformanceEngine(
    reader_pv, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_pv.run_audit(context_name="ETL-02-PV-Ingest-Casting")

In [0]:
# COMMAND ----------
print("🕵️ Rozpoczynam audyt dla procesu: Inverter logs (ETL-04)...")

TABLE_INVERTER = "workspace.bronze.inverter_logs_partition_nightmare"
df_inverter = spark.read.table(TABLE_INVERTER)

reader_inverter = DataFrameExplainReader(spark, df=df_inverter, table_name=TABLE_INVERTER)

engine_inverter = PerformanceEngine(
    reader_inverter, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_inverter.run_audit(context_name="ETL-04-Inverter-OverPartitioning")

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 5] Rozpoczynam audyt: Cartesian Product Detection (ETL-05)...")

# Generate two independent DataFrames in-memory
df_alpha = spark.range(1, 10).withColumnRenamed("id", "id_alpha")
df_beta = spark.range(1, 10).withColumnRenamed("id", "id_beta")

# ANTI-PATTERN: Cross Join (missing join condition)
df_cartesian = df_alpha.crossJoin(df_beta)

# Przekazujemy DataFrame do czytnika (analiza planu in-memory)
reader_cartesian = DataFrameExplainReader(spark, df=df_cartesian)

engine_cartesian = PerformanceEngine(
    reader_cartesian, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_cartesian.run_audit(context_name="ETL-05-Cartesian-Nightmare")

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 6] Rozpoczynam audyt: Data Skew Analysis (ETL-06)...")

TABLE_INVERTER = "workspace.bronze.inverter_logs_partition_nightmare"
TABLE_STATIONS = "workspace.bronze.stations_dict"

try:
    df_inv = spark.read.table(TABLE_INVERTER)
    df_stat = spark.read.table(TABLE_STATIONS)

    # ROBUST SOLUTION FOR SERVERLESS:
    # Instead of modifying locked global session configuration (which throws AnalysisException),
    # force SortMergeJoin at specific relation level using local hint .hint("merge").
    # This way Catalyst will skip Broadcast and generate Shuffle Exchange plan prone to Data Skew.
    df_skewed_join = df_inv.hint("merge").join(
        df_stat, df_inv.inverter_id == df_stat.station_id, "inner"
    )

    reader_skew = DataFrameExplainReader(spark, df=df_skewed_join)
    engine_skew = PerformanceEngine(
        reader_skew, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
    )
    engine_skew.run_audit(context_name="ETL-06-Data-Skew-Analysis")

except Exception as e:
    print(f"❌ [AUDYT-6-ERROR] Failed to perform skew audit: {str(e)}")

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 7] Rozpoczynam audyt: File System Indexing Verification (ETL-07)...")

TABLE_PV = "workspace.bronze.pv_metrics_string_nightmare"
df_pv_filtered = spark.read.table(TABLE_PV).filter("str_temperature > '30.0'")

# Pass artificial metric num_files > 50 through dictionary for rule invocation
reader_missing_idx = DataFrameExplainReader(spark, df=df_pv_filtered, table_name=TABLE_PV)

# Fetch base metrics and simulate large number of files in metadata
mock_metrics = reader_missing_idx.get_physical_metrics()
mock_metrics["num_files"] = 120  # Force threshold exceeded 50 files

# Override engine call with injected laboratory metrics
plan = reader_missing_idx.get_execution_plan()
alert = MissingOptimizationRule().evaluate(plan, mock_metrics, policy_mgr.get_policy("performance"))

print("\n" + "=" * 60)
print("📊 [APM MANUAL REPORT] Context: ETL-07-Missing-Cluster-Index")
print("=" * 60)
if alert:
    print(
        f"🚨 ALERT: [{alert.rule_id}] - {alert.title}\n📋 Description: {alert.description}\n💡 Fix: {alert.fix}"
    )
else:
    print("✅ File system is optimized.")
print("=" * 60)

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 8] Rozpoczynam audyt: Explode Function Abuse Analysis (ETL-08)...")

from pyspark.sql import functions as F

# Create DataFrame containing nested structures (Array)
df_arrays = spark.range(1, 10).withColumn(
    "telemetry_chunks", F.array(F.lit(100), F.lit(200), F.lit(300))
)

# ANTY-WZORZEC: Explode generating drastic row increase in RAM
df_exploded = df_arrays.withColumn("exploded_chunk", F.explode(F.col("telemetry_chunks")))

reader_explode = DataFrameExplainReader(spark, df=df_exploded)
engine_explode = PerformanceEngine(
    reader_explode, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_explode.run_audit(context_name="ETL-08-Explode-Abuse")

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 9] Rozpoczynam audyt: Wielokrotne Skanowanie Tabelem (ETL-09)...")

df_base = spark.read.table("workspace.bronze.bess_telemetry_small_files")

# ANTY-WZORZEC: Wielokrotne joinowanie tego samego DataFrame bez zapisu stanu (Cache)
df_redundant = df_base.join(df_base, "id").join(df_base, "id")

reader_redundant = DataFrameExplainReader(spark, df=df_redundant)
engine_redundant = PerformanceEngine(
    reader_redundant, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_redundant.run_audit(context_name="ETL-09-Redundant-Scan")

In [0]:
# COMMAND ----------
print("🕵️ [AUDYT 10] Rozpoczynam audyt: UDF Function Performance Analysis (ETL-10)...")

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType


# Telemetry pipeline with black box (Standard Python UDF)
@udf(returnType=DoubleType())
def standard_python_conversion(voltage_str):
    try:
        return float(voltage_str) * 1.5
    except:
        return 0.0


# Apply black box to PV table
df_pv_raw = spark.read.table("workspace.bronze.pv_metrics_string_nightmare")
df_udf = df_pv_raw.withColumn("converted_voltage", standard_python_conversion(F.col("str_voltage")))

reader_udf = DataFrameExplainReader(spark, df=df_udf)
engine_udf = PerformanceEngine(
    reader_udf, active_rules, policy_mgr, env_prov, remedy_eng, cost_trans, reporter
)
engine_udf.run_audit(context_name="ETL-10-Python-UDF-Abuse")